### 3. Yếu tố Nhân sự & Thời gian
Vai trò của Đạo diễn/diễn viên có tên tuổi lớn có tương quan mạnh mẽ với rating người dùng cao hơn không? Năm sản xuất (phim cũ/mới) có ảnh hưởng đến rating trung bình không?

- Input: director_power, actor_power, release_year.
- Output: cột Rating thực tế.

In [ ]:
# # Chuẩn bị dữ liệu cho mô hình
# # Chỉ lấy 3 cột bạn quan tâm: Đạo diễn, Diễn viên, Năm sản xuất
features = ['director_power', 'actor_power', 'release_year']
X = full_data[features]
y = full_data['avg_rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Đang huấn luyện mô hình...")
model = xgb.XGBRegressor(
    n_estimators=1000,   # Tăng số cây lên (vì ta sẽ dừng sớm nếu nó học xong)
    learning_rate=0.05,  # Học chậm lại chút cho chắc
    max_depth=4, 
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50 # QUAN TRỌNG: Nếu sau 50 cây mà không khôn hơn thì dừng lại (Chống học vẹt)
)

# Đưa cả tập Test vào để mô hình tự kiểm tra (eval_set)
model.fit(
    X_train, y_train, 
    eval_set=[(X_test, y_test)], 
    verbose=False
)

# Kiểm tra sai số
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"Sai số trung bình (RMSE) trên tập Test: {rmse:.4f}")
# Nếu RMSE ~ 0.8 - 1.0 là ổn (vì thang điểm 5, lệch 1 sao là chấp nhận được)

#### 3. Phân tích SHAP

SHAP (SHapley Additive exPlanations) là một kỹ thuật xuất phát từ Lý thuyết Trò chơi (Game Theory), dùng để giải thích các mô hình "hộp đen" phức tạp như XGBoost.

SHAP có nhiệm vụ chia nhỏ kết quả dự đoán (Rating) thành các phần đóng góp công bằng của từng yếu tố (Đạo diễn, Diễn viên, Tiền...), giúp chúng ta hiểu được mô hình đang "nghĩ" gì.

In [ ]:
# Tính giá trị SHAP để xem mức độ tác động
explainer = shap.Explainer(model)
shap_values = explainer(X)

#### 4. Vẽ biểu đồ (SHAP Dependence Plots)

Trục Hoành (X-axis): Giá trị thực tế của dữ liệu (Ví dụ: Điểm lịch sử của Đạo diễn là 3.0 hay 4.0).

Trục Tung (Y-axis - SHAP value): Mức độ tác động lên điểm Rating dự đoán.

+ Giá trị > 0: Yếu tố này giúp Cộng điểm cho phim (Kéo Rating lên).

+ Giá trị < 0: Yếu tố này làm Trừ điểm phim (Kéo Rating xuống).

+ Giá trị = 0: Không ảnh hưởng.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(24, 6))

# Câu hỏi 1: ĐẠO DIỄN có tên tuổi lớn (Rating lịch sử cao) có kéo điểm phim lên không?
shap.dependence_plot("director_power", shap_values.values, X, interaction_index=None, ax=ax[0], show=False)
ax[0].set_title("Tác động của DANH TIẾNG ĐẠO DIỄN", fontsize=14, fontweight='bold')
ax[0].set_xlabel("Điểm trung bình lịch sử của Đạo diễn (Director Power)")
ax[0].set_ylabel("Mức độ tăng/giảm Rating dự báo")
ax[0].grid(True, alpha=0.3)

# Câu hỏi 2: DIỄN VIÊN có tên tuổi lớn có kéo điểm phim lên không?
shap.dependence_plot("actor_power", shap_values.values, X, interaction_index=None, ax=ax[1], show=False)
ax[1].set_title("Tác động của DANH TIẾNG DIỄN VIÊN", fontsize=14, fontweight='bold')
ax[1].set_xlabel("Điểm trung bình lịch sử của Diễn viên (Actor Power)")
ax[1].grid(True, alpha=0.3)

# Câu hỏi 3: NĂM SẢN XUẤT (Phim cũ vs Mới) ảnh hưởng thế nào?
shap.dependence_plot("release_year", shap_values.values, X, interaction_index=None, ax=ax[2], show=False)
ax[2].set_title("Tác động của THỜI GIAN (NĂM SẢN XUẤT)", fontsize=14, fontweight='bold')
ax[2].set_xlabel("Năm phát hành")
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### 5. Tầm quan trọng

In [ ]:
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.title("Yếu tố nào quan trọng nhất trong 3 yếu tố này?")
plt.show()

#### 6. Các phân tích liên quan



In [ ]:
# Chia nhóm (Binning) Power thành 4 mức: Low, Medium, High, Legendary

full_data['Director_Tier'] = pd.qcut(full_data['director_power'], q=4, labels=['Low', 'Mid', 'High', 'Legendary'])

full_data['Actor_Tier'] = pd.qcut(full_data['actor_power'], q=4, labels=['Low', 'Mid', 'High', 'Legendary'])

In [ ]:
# Tạo bảng Pivot
pivot_table = full_data.pivot_table(index='Director_Tier', columns='Actor_Tier', values='Rating', aggfunc='mean')

In [ ]:
# Vẽ Heatmap

plt.figure(figsize=(8, 6))

sns.heatmap(pivot_table, annot=True, fmt=".2f", cmap="YlGnBu")

plt.title("Ma trận cộng hưởng: Đạo diễn vs Diễn viên")

plt.show()

In [ ]:
# --- BƯỚC 1: TẠO MOVIE_SUMMARY (KHAI BÁO LẠI CHO CHẮC) ---
# Gom nhóm full_data thành dữ liệu từng phim
movie_summary = full_data.groupby('Title').agg({
    'avg_rating': 'mean',
    'director_power': 'first',
    'director': 'first',
    'Genres': 'first',
    'release_year': 'first',
    'vote_count': 'first'
}).reset_index()

# Lọc bỏ phim ít vote để tránh nhiễu
movie_summary = movie_summary[movie_summary['vote_count'] > 50]

# --- BƯỚC 2: TÍNH ĐỘ LỆCH (DIFF) ---
# Diff = Rating Thực tế - Kỳ vọng (Power)
movie_summary['Diff'] = movie_summary['avg_rating'] - movie_summary['director_power']

# --- BƯỚC 3: LỌC BOM XỊT & HIỆN TƯỢNG ---

# 1. Bom xịt: Kỳ vọng cao (Power > 3.5) nhưng Lệch âm lớn nhất (Thực tế kém xa kỳ vọng)
flops = movie_summary[movie_summary['director_power'] > 3.5].sort_values('Diff', ascending=True).head(10)

# 2. Hiện tượng: Kỳ vọng thấp (Power < 3.5) nhưng Lệch dương lớn nhất (Thực tế vượt xa kỳ vọng)
breakouts = movie_summary[movie_summary['director_power'] < 3.5].sort_values('Diff', ascending=False).head(10)

# --- BƯỚC 4: IN KẾT QUẢ ---
cols = ['Title', 'director', 'avg_rating', 'director_power', 'Diff']

print("\n=== TOP 10 SA SÚT (Gây thất vọng nhất) ===")
print(flops[cols].to_string(index=False))

print("\n=== TOP 10 BỨT PHÁ (Gây bất ngờ nhất) ===")
print(breakouts[cols].to_string(index=False))

# --- BƯỚC 5: TRỰC QUAN HÓA ---
plt.figure(figsize=(12, 8))
# Vẽ nền
sns.scatterplot(data=movie_summary, x='director_power', y='avg_rating', color='lightgrey', alpha=0.5, label='Phim khác')
# Vẽ Bom xịt
sns.scatterplot(data=flops, x='director_power', y='avg_rating', color='red', s=100, label='Sa sút (Diff Âm lớn)')
# Vẽ Hiện tượng
sns.scatterplot(data=breakouts, x='director_power', y='avg_rating', color='green', s=100, label='Bứt phá (Diff Dương lớn)')

plt.title('Phân tích Ngoại lệ dựa trên ĐỘ LỆCH (Diff)', fontsize=15)
plt.xlabel('Sức mạnh Đạo diễn (Kỳ vọng)')
plt.ylabel('Rating Thực tế')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- BƯỚC 1: TẠO NHÃN (LABELING) DỰA TRÊN LOGIC "ĐỘ LỆCH" (DIFF) ---
# Sử dụng lại movie_summary từ bước trước

# 1. Xác định ngưỡng (Dùng Quantile để linh hoạt)
high_power_thresh = movie_summary['director_power'].quantile(0.80) # Top 20% quyền lực
low_power_thresh = movie_summary['director_power'].quantile(0.50)  # Bottom 50%
diff_low_thresh = movie_summary['Diff'].quantile(0.20)  # Top 20% gây thất vọng nhất
diff_high_thresh = movie_summary['Diff'].quantile(0.80) # Top 20% gây bất ngờ nhất

# 2. Hàm dán nhãn
def get_movie_fate(row):
    # Nhóm Sa sút (Flop): Đạo diễn xịn + Diff Âm lớn
    if (row['director_power'] >= high_power_thresh) and (row['Diff'] <= diff_low_thresh):
        return 0 # Mã 0: SA SÚT
    
    # Nhóm Bứt phá (Breakout): Đạo diễn thường + Diff Dương lớn
    elif (row['director_power'] <= low_power_thresh) and (row['Diff'] >= diff_high_thresh):
        return 1 # Mã 1: BỨT PHÁ
    
    # Còn lại là Bình thường
    else:
        return 2 # Mã 2: BÌNH THƯỜNG

# Áp dụng hàm để tạo cột mục tiêu (Target)
movie_summary['Fate'] = movie_summary.apply(get_movie_fate, axis=1)

print("Phân phối nhãn dự báo:")
print(movie_summary['Fate'].value_counts())
# Bạn sẽ thấy nhóm 0 và 1 khá ít, nhóm 2 chiếm đa số -> Đây là bài toán mất cân bằng dữ liệu

# --- BƯỚC 2: CHUẨN BỊ DỮ LIỆU HUẤN LUYỆN (FEATURES) ---
# Input (X): Chỉ dùng thông tin CÓ TRƯỚC khi phim chiếu
# TUYỆT ĐỐI KHÔNG dùng 'Rating' hay 'Diff' trong X (vì đó là đáp án)
feature_cols = ['director_power', 'release_year'] 
# Nếu bạn có thêm budget hay genres ở movie_summary thì thêm vào đây sẽ tốt hơn

X = movie_summary[feature_cols]
y = movie_summary['Fate']

# Chia tập Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- BƯỚC 3: HUẤN LUYỆN MÔ HÌNH ---
# class_weight='balanced': Cực kỳ quan trọng để máy chú ý vào nhóm thiểu số (Sa sút/Bứt phá)
print("\nĐang huấn luyện mô hình dự báo...")
clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

# --- BƯỚC 4: ĐÁNH GIÁ & TRỰC QUAN HÓA ---
y_pred = clf.predict(X_test)

# In báo cáo
target_names = ['Sa sút (Flop)', 'Bứt phá (Breakout)', 'Bình thường']
print("\n" + "="*40)
print("KẾT QUẢ DỰ BÁO:")
print("="*40)
print(classification_report(y_test, y_pred, target_names=target_names))

# Vẽ Ma trận nhầm lẫn (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', 
            xticklabels=target_names, yticklabels=target_names)
plt.title('Ma trận nhầm lẫn: Dự báo Sa sút & Bứt phá')
plt.ylabel('Thực tế (Dựa trên Diff)')
plt.xlabel('Máy dự báo')
plt.show()